# Day 21 — Build a simple agent (hands-on)

One model, **one tool**, the loop from Day 19: a help-desk agent that *decides when to search
a knowledge base* instead of always retrieving. Runs against the Anthropic API when
`ANTHROPIC_API_KEY` is set, with a working local fallback so it executes offline.

## Agenda (60 min)

| # | Segment | Time |
| - | ------- | ---- |
| 0 | The task: agentic RAG with one tool | 4 min |
| 1 | The tool: `search_kb` | 8 min |
| 2 | The agent loop (Anthropic tool-use + local fallback) | 16 min |
| 3 | Run it: single-hop, multi-hop, no-match | 12 min |
| 4 | Guards in action | 8 min |
| 5 | Evaluate: agentic vs always-retrieve | 9 min |
| 6 | Exercises + quiz | 3 min |

Kernel: **Python (ai-upskill)**.

In [1]:
import os, re, json, time, warnings
warnings.filterwarnings("ignore")
import numpy as np
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2")
HAVE_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
print("ANTHROPIC_API_KEY set:", HAVE_ANTHROPIC)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 22654.08it/s]

ANTHROPIC_API_KEY set: False


## 0 — The task (4 min)

A support assistant. Day 18's RAG *always* retrieves once. That's wasteful for "hi" and
"thanks", and insufficient for "compare the refund and cancellation policies" (needs two
lookups). An **agent** with a `search_kb` tool decides:

- greeting / chit-chat → answer directly, no search
- one factual question → one search
- multi-part question → several searches
- out of scope → search, see nothing relevant, say so

Same loop as Day 19; the only tool is retrieval.

## 1 — The tool (8 min)

In [2]:
KB = {
 "refund": "Refunds are issued to the original payment method within 5 business days of approval. Digital goods are non-refundable once downloaded.",
 "cancel": "You can cancel your subscription anytime in billing settings. Cancellation stops future charges; the current period is not prorated.",
 "shipping": "Standard shipping is free over $50 and takes 3-5 business days in the US. Expedited is $15, 1-2 days. US and Canada only.",
 "password": "Reset your password with the 'forgot password' link. The reset link expires in 1 hour. Accounts lock after 5 failed attempts for 30 minutes.",
 "api_limits": "The API allows 600 requests per minute per key. Exceeding it returns HTTP 429 with a Retry-After header.",
 "data_export": "Export your data as CSV or JSON from Account > Data. Large exports are emailed as a download link within 24 hours.",
 "sla": "Enterprise plans include a 99.9% monthly uptime SLA with service credits for breaches. Standard plans have no SLA.",
}
KB_KEYS = list(KB)
KB_VEC = embedder.encode(list(KB.values()), normalize_embeddings=True)

def search_kb(query, k=2):
    # the agent's only tool: returns up to k KB passages with relevance scores
    qv = embedder.encode([query], normalize_embeddings=True)[0]
    sims = KB_VEC @ qv
    out = []
    for i in np.argsort(-sims)[:k]:
        out.append({"topic": KB_KEYS[i], "score": round(float(sims[i]), 2), "text": KB[KB_KEYS[i]]})
    return out

SEARCH_TOOL = {
 "name": "search_kb",
 "description": "Search the support knowledge base for relevant passages. Call it once per "
                "distinct thing you need to look up. Returns passages with a relevance score "
                "(>0.4 is a good match; <0.3 means nothing relevant).",
 "input_schema": {"type": "object",
                  "properties": {"query": {"type": "string", "description": "what to look up"}},
                  "required": ["query"]},
}
print(json.dumps(search_kb("how long do refunds take"), indent=1))

[
 {
  "topic": "refund",
  "score": 0.73,
  "text": "Refunds are issued to the original payment method within 5 business days of approval. Digital goods are non-refundable once downloaded."
 },
 {
  "topic": "shipping",
  "score": 0.27,
  "text": "Standard shipping is free over $50 and takes 3-5 business days in the US. Expedited is $15, 1-2 days. US and Canada only."
 }
]


## 2 — The agent loop (16 min)

### The real Anthropic tool-use loop

```python
import anthropic
client = anthropic.Anthropic()

def run_agent_anthropic(question, max_steps=5):
    messages = [{"role": "user", "content": question}]
    for _ in range(max_steps):
        resp = client.messages.create(
            model="claude-opus-5", max_tokens=600, system=SYSTEM,
            tools=[SEARCH_TOOL], messages=messages)
        messages.append({"role": "assistant", "content": resp.content})
        if resp.stop_reason != "tool_use":
            return "".join(b.text for b in resp.content if b.type == "text")
        results = []
        for b in resp.content:
            if b.type == "tool_use":                      # b.name == "search_kb"
                out = search_kb(**b.input)
                results.append({"type": "tool_result", "tool_use_id": b.id,
                                "content": json.dumps(out)})
        messages.append({"role": "user", "content": results})
    return "stopped: max_steps"
```

Below: the same loop, with a **local planner** standing in for the model when there's no key.

In [3]:
SYSTEM = (
 "You are a support assistant with one tool: search_kb. Decide whether you need it. "
 "For greetings or small talk, answer directly without searching. For factual questions, "
 "search for each distinct thing you need, then answer ONLY from the returned passages and "
 "cite the topic(s). If the best score is below 0.3, say you don't have that information."
)

# -------- local planner: a deterministic stand-in for the model's decisions --------
GREETING = re.compile(r"^\s*(hi|hey|hello|thanks|thank you|good morning|bye)\b", re.I)

def local_planner(question, history):
    q = question.lower()
    searches_done = [h for h in history if h[0] == "search_kb"]
    if GREETING.match(q) and not searches_done:
        return {"type": "final", "text": "Hi! How can I help with your account, billing, or the API?"}
    # split multi-part questions on 'and' / 'compare ... and ...'
    parts = re.split(r"\band\b|;|,", question) if ("compare" in q or " and " in q) else [question]
    parts = [p.strip() for p in parts if len(p.strip()) > 8]
    if len(searches_done) < len(parts):
        return {"type": "action", "tool": "search_kb", "input": {"query": parts[len(searches_done)]}}
    # synthesize from everything found
    passages = [r for h in searches_done for r in h[2]]
    good = [p for p in passages if p["score"] >= 0.3]
    if not good:
        return {"type": "final", "text": "I don't have information on that. Try rephrasing, "
                "or contact support@example.com."}
    seen, uniq = set(), []
    for p in sorted(good, key=lambda x: -x["score"]):
        if p["topic"] not in seen:
            seen.add(p["topic"]); uniq.append(p)
    answer = " ".join(p["text"] for p in uniq[:len(parts)])
    return {"type": "final", "text": f"{answer}  [sources: {', '.join(sorted(seen))}]"}

def run_agent(question, planner=local_planner, max_steps=5, verbose=True):
    history = []
    for step in range(max_steps):
        act = planner(question, history)
        if act["type"] == "final":
            if verbose: print(f"[{step}] FINAL")
            return dict(answer=act["text"], steps=step, searches=[h[1]["query"] for h in history])
        result = search_kb(**act["input"])
        history.append(("search_kb", act["input"], result))
        if verbose: print(f"[{step}] search_kb({act['input']['query']!r}) -> "
                          f"{[(r['topic'], r['score']) for r in result]}")
    return dict(answer="stopped: max_steps", steps=max_steps, searches=[h[1]['query'] for h in history])

run_agent = run_agent  # (swap planner=... for the Anthropic version when you have a key)

## 3 — Run it (12 min)

In [4]:
for q in ["hi there",
          "how long do refunds take?",
          "compare the refund policy and the cancellation policy",
          "what's the API rate limit and how do I export my data?",
          "what is the meaning of life"]:
    print(f"\nQ: {q}")
    out = run_agent(q)
    print(f"  searches: {out['searches']}")
    print(f"  A: {out['answer'][:200]}")


Q: hi there
[0] FINAL
  searches: []
  A: Hi! How can I help with your account, billing, or the API?

Q: how long do refunds take?
[0] search_kb('how long do refunds take?') -> [('refund', 0.75), ('shipping', 0.3)]
[1] FINAL
  searches: ['how long do refunds take?']
  A: Refunds are issued to the original payment method within 5 business days of approval. Digital goods are non-refundable once downloaded.  [sources: refund, shipping]

Q: compare the refund policy and the cancellation policy
[0] search_kb('compare the refund policy') -> [('refund', 0.51), ('sla', 0.22)]
[1] search_kb('the cancellation policy') -> [('cancel', 0.5), ('refund', 0.3)]
[2] FINAL
  searches: ['compare the refund policy', 'the cancellation policy']
  A: Refunds are issued to the original payment method within 5 business days of approval. Digital goods are non-refundable once downloaded. You can cancel your subscription anytime in billing settings. Ca

Q: what's the API rate limit and how do I export my data?
[

Notice the agent made **0 searches** for the greeting and the philosophy question (well — one
low-score search then abstained), **1** for the simple factual question, and **2** for each
multi-part question. A fixed RAG pipeline would have done exactly one every time.

## 4 — Guards in action (8 min)

In [5]:
def looping_planner(question, history):
    # a buggy planner that keeps searching the same thing
    return {"type": "action", "tool": "search_kb", "input": {"query": "refund"}}

out = run_agent("how do refunds work", planner=looping_planner, max_steps=4, verbose=False)
print("no guard :", out["answer"], "| searches:", len(out["searches"]))

def run_agent_guarded(question, planner, max_steps=5, max_same=2, verbose=False):
    history, counts = [], {}
    for step in range(max_steps):
        act = planner(question, history)
        if act["type"] == "final":
            return dict(answer=act["text"], steps=step)
        key = act["input"]["query"]
        counts[key] = counts.get(key, 0) + 1
        if counts[key] > max_same:
            return dict(answer=f"ABORTED: searched {key!r} {counts[key]}x without progress", steps=step)
        history.append(("search_kb", act["input"], search_kb(**act["input"])))
    return dict(answer="stopped: max_steps", steps=max_steps)

print("guarded  :", run_agent_guarded("how do refunds work", looping_planner)["answer"])

no guard : stopped: max_steps | searches: 4
guarded  : ABORTED: searched 'refund' 3x without progress


## 5 — Evaluate: agentic vs always-retrieve (9 min)

Mix KB questions, multi-part questions, greetings, and out-of-scope. Score answer correctness,
whether greetings/OOS are handled without a wasted search, and mean searches per query (cost).

In [6]:
EVAL = [
 ("how long do refunds take", "5 business days", True),
 ("does cancelling prorate the current period", "not prorated", True),
 ("what HTTP code means rate limited", "429", True),
 ("compare refund and cancellation policies", "5 business days", True),  # needs 2 hops
 ("what's the API limit and how are large exports delivered", "600 requests per minute", True),
 ("hello", None, False),
 ("thanks for your help", None, False),
 ("what is your company's valuation", None, False),
 ("do you sell gift cards", None, False),
]

def always_retrieve(question):
    # the Day 18 baseline: one search, then answer
    res = search_kb(question, k=2)
    good = [r for r in res if r["score"] >= 0.3]
    ans = (" ".join(r["text"] for r in good) + f"  [sources: {', '.join(r['topic'] for r in good)}]") \
          if good else "I don't have information on that."
    return dict(answer=ans, searches=[question])

def score(runner):
    correct, wasted_search, searches = 0, 0, []
    for q, fact, is_kb in EVAL:
        o = runner(q)
        searches.append(len(o["searches"]))
        if is_kb:
            correct += int(fact and fact.lower() in o["answer"].lower())
        else:
            correct += int("don't have" in o["answer"].lower() or "how can i help" in o["answer"].lower()
                           or "you're welcome" in o["answer"].lower() or o["answer"].startswith("Hi"))
            wasted_search += int(len(o["searches"]) > 0)
    return dict(correctness=correct/len(EVAL), wasted_searches=wasted_search,
                mean_searches=float(np.mean(searches)))

print("agentic       :", score(lambda q: run_agent(q, verbose=False)))
print("always-retrieve:", score(always_retrieve))

agentic       : {'correctness': 1.0, 'wasted_searches': 2, 'mean_searches': 1.0}


always-retrieve: {'correctness': 1.0, 'wasted_searches': 4, 'mean_searches': 1.0}


Result on this small set: equal correctness, but the agent **halves the wasted searches** —
it does zero for greetings and small talk, where always-retrieve fires a pointless lookup
every time. The correctness gap opens up on a **larger KB**, where the multi-part questions
genuinely need two separate retrievals and always-retrieve's single lookup misses the second
half (try it — Exercise-style, add 20 more KB entries). Against the Anthropic API the gap
widens further: the model splits multi-part questions better and phrases abstentions correctly.

**The tradeoff:** the agent makes 1–3 model calls per query instead of 1 (Day 19's
super-linear cost). It's worth it when queries vary a lot in what they need; it's overhead
when every query is "retrieve once and answer".

## 6 — Exercises

1. **Wire the real API.** If you have a key, implement `run_agent_anthropic` from §2 and run
   the §3 questions through it. Compare `searches` and answers to the local planner.
2. **Add a second tool.** Add `get_order_status(order_id)` returning a canned dict. Give the
   agent a question that needs both tools ("did my refund for order 5512 go through, and how
   long should it take?").
3. **Confidence-aware abstention.** Change the planner to abstain when the *best* score across
   all searches is < 0.35 (not 0.3). Re-score. Does OOS handling improve without hurting KB
   correctness?
4. **Multi-hop that the splitter misses.** Write a question whose two lookups aren't joined by
   "and" (e.g. "I was charged after I cancelled — what should have happened?"). Show the local
   planner does one search; describe how a real model would plan it.
5. **Cost accounting.** Assume 1 model call = 900 input + 150 output tokens, +400 input per
   prior tool result, `claude-haiku-4-5` pricing. Compute $/query for a 1-hop vs a 3-hop
   agent run. At 100k queries/month, what's the delta?
6. **Loop + budget guard together.** Combine `run_agent_guarded` (repeat detection) with a
   `max_searches=4` total budget and a `max_steps=6`. Feed it the looping planner and a
   planner that searches 6 distinct things; show both are contained.

> **Attempt every exercise and the quiz first.** The worked solutions and the answer key live in [`solutions/solutions.ipynb`](solutions/solutions.ipynb) — open it only to check your work, not to start.

## Self-check quiz


1. What decision does an agentic RAG system make that a fixed RAG pipeline doesn't?
2. In the Anthropic loop, how do you know the model is done vs wants another tool call?
3. Why give the tool description a note about what score means "no match"?
4. Name two guards this agent needs and what each prevents.
5. When is an agent's per-query cost worth it over always-retrieve?
6. Your agent loops, searching the same query repeatedly. Which guard catches it?
7. A multi-part question comes in with no "and". What does a keyword-splitting planner do, and
   what would a real model do?

## Where this goes next

Week 7 done — you can build and reason about agents. **Week 8 — APIs in Practice**: the
Anthropic SDK in depth — message structure, parameters, function calling internals, streaming
(Day 22).